# Required Libraries and Functions

In [0]:
import json
from pyspark.sql.functions import col, current_timestamp, when, date_trunc, expr, broadcast, lead
from pyspark.sql.window import Window
from pyspark.sql.types import DateType

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/taxi_weather/datasets/")

base_path = dbutils.widgets.get("volume_path")

In [0]:
dates_file_path = f"{base_path}pipeline_dates.json"

with open(dates_file_path, "r") as f:
    date_params = json.load(f)

str_date = date_params["start_date"]
end_date = date_params["end_date"]


# Load and Preprocess Weather Data

In [0]:
weather_df = spark.read.table("workspace.taxi_weather.bronze_weather")
wc_df = spark.read.table("workspace.taxi_weather.silver_wc_dim")
borough_df = spark.read.table("workspace.taxi_weather.silver_borough_dim")

window_spec = Window.partitionBy("bd.borough_id").orderBy("weather_datetime")

clean_weather = (weather_df.alias("wd")
                .join(broadcast(borough_df).alias("bd"), how="left", on=col("wd.borough")==col("bd.name"))
                .join(broadcast(wc_df).alias("wc"), how="left", on=col("wd.weather_code")==col("wc.weather_code"))
                .withColumn("weather_datetime", (col("weather_datetime")).cast("timestamp_ntz"))
                .withColumn("precipitation", lead("precipitation", 1).over(window_spec))
                .withColumn("snowfall", lead("snowfall", 1).over(window_spec))
                )

silver_weather = (
    clean_weather.select(
        col("weather_datetime"), 
        col("wd.weather_code"),
        col("wc.description"),
        col("bd.borough_id"),
        col("temperature_2m"), 
        col("precipitation"), 
        col("snowfall")
        )
        .withColumn("silver_processed_time", current_timestamp())
        .where((col("weather_datetime") >= str_date) & (col("weather_datetime") < end_date))
        )

# Load and Preprocess Zone Data

In [0]:
zone_df = spark.read.table("workspace.taxi_weather.bronze_zone")
borough_df = spark.read.table("workspace.taxi_weather.silver_borough_dim")

clean_zone_1 = (zone_df.withColumn("Borough",
                                   when(col("Borough").isin("N/A", "Unknown"), "Unknown")
                                   .otherwise(col("Borough")))
                .withColumn("Zone",
                            when(col("Zone") == "N/A", "Unknown")
                            .otherwise(col("Zone")))
                .withColumn("service_zone",
                            when(col("service_zone") == "N/A", "Unknown")
                            .otherwise(col("service_zone")))
                )
                        

clean_zone_2 = (clean_zone_1.alias("cz1")
                .join(broadcast(borough_df).alias("bd"), how="left", on=col("cz1.Borough")==col("bd.name"))
                )

silver_zone = (
    clean_zone_2.select(
        col("LocationID"),
        col("bd.borough_id"),
        col("Zone"),
        col("service_zone")
        )
    .withColumn("silver_processed_time", current_timestamp())
    )


# Load, Preprocess and Quarintine Trips Data

In [0]:
trips_df = spark.read.table("workspace.taxi_weather.bronze_trips")

base_trips = (trips_df
              .fillna(0, subset=["passenger_count", "Airport_fee", "congestion_surcharge"])
              .withColumn("RatecodeID",
                          when(col("RatecodeID").isNull(), 99)
                          .otherwise(col("RatecodeID"))
                          )
              .withColumn("store_and_fwd_flag",
                          when(col("store_and_fwd_flag").isNull(), "Unknown")
                          .otherwise(col("store_and_fwd_flag"))
                          )
              .withColumn("pickup_datehour", date_trunc("hour", col("tpep_pickup_datetime")).cast("timestamp_ntz"))
              .withColumn("dropoff_datehour", date_trunc("hour", col("tpep_dropoff_datetime")).cast("timestamp_ntz"))
              )

tagged_trips = (base_trips
                .withColumn("rejection_reason",
                            when((col("tpep_pickup_datetime") < str_date) | (col("tpep_pickup_datetime") >= end_date), "Out of bounds date")
                            .when(col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime"), "Pickup after dropoff")
                            .when(col("trip_distance") <= 0, "Invalid distance")
                            .when(col("fare_amount") <= 0, "Invalid fare")
                            .when(col("extra") < 0, "Negative extra")
                            .otherwise("Valid")
                            )
                )

clean_trips = (tagged_trips.where(col("rejection_reason") == "Valid"))

silver_trips = (
    clean_trips.select(
        col("pickup_datehour"),
        col("dropoff_datehour"),
        
        col("tpep_pickup_datetime"),
        col("tpep_dropoff_datetime"),
        col("PULocationID"),
        col("DOLocationID"),

        col("passenger_count"),
        col("trip_distance"),

        col("store_and_fwd_flag"),
        col("VendorID"),
        col("RatecodeID"),
        col("payment_type"),

        col("fare_amount"),
        col("extra"),
        col("mta_tax"),
        col("tip_amount"),
        col("tolls_amount"),
        col("improvement_surcharge"),
        col("total_amount"),
        col("congestion_surcharge"),
        col("Airport_fee"),
        col("cbd_congestion_fee")
        )
    .withColumn("silver_processed_time", current_timestamp())
    )
    
quarantine_trips = (tagged_trips
                    .where(col("rejection_reason") != "Valid")
                    .withColumn("silver_processed_time", current_timestamp())
                    )

# Write to Delta Tables

In [0]:
silver_weather.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.silver_weather")

silver_zone.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.silver_zone")

silver_trips.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.silver_trips")

quarantine_trips.write.format("delta").mode("overwrite").saveAsTable("workspace.taxi_weather.silver_quarantine_trips")